In [ ]:
# === CELL 1: SETUP & CONFIGURATION ===
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm  # For MobileNetV3 Backbone
import numpy as np
import os
import cv2
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast # Crucial for 3080 Ti speed

# Hardware Check
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🚀 Running on: {DEVICE} (AMP Enabled)")

# GLOBAL CONFIGURATION
CONFIG = {
    # Input
    'img_height': 480,
    'img_width': 640,
    'input_channels': 1, # Grayscale
    
    # Stereo Geometry
    'max_disp_pixel': 192,         # Max disparity in original image pixels
    'backbone_stride': 4,          # MobileNet Stage 1 output
    'internal_disp_steps': 48,     # 192 / 4 = 48 steps for cost volume
    
    # Heads
    'num_seg_classes': 6,          # 5 Robotics + Background
    'num_det_classes': 80,         # COCO standard
    
    # Training
    'batch_size': 8,               # Adjust for 3080 Ti (12GB VRAM)
    'lr_backbone': 1e-4,
    'lr_heads': 3e-4,
}

print(f"ℹ️  Configuration Loaded. Max Disparity: {CONFIG['max_disp_pixel']}px ({CONFIG['internal_disp_steps']} internal steps)")

In [ ]:
# === CELL 2: DATA LOADING PIPELINE ===
class FusedHexapodDataset(Dataset):
    def __init__(self, root_dir, mode='train', task='stereo', transform=None):
        """
        mode: 'train' or 'val'
        task: 'stereo' (FT3D), 'robotics' (TartanAir), 'coco' (Detection)
        """
        self.root = root_dir
        self.mode = mode
        self.task = task
        self.transform = transform
        self.file_list = self._scan_files()

    def _scan_files(self):
        # Placeholder: Implement actual file scanning logic here
        # Return list of dicts: [{'left': path, 'right': path, 'disp': path, 'seg': path, ...}]
        return [] 

    def load_disp(self, path):
        # Handle .pfm or .png disparity loading
        if path.endswith('.pfm'):
            # Custom PFM loader would go here
            return np.zeros((480, 640), dtype=np.float32)
        return cv2.imread(path, cv2.IMREAD_UNCHANGED).astype(np.float32) / 256.0

    def __len__(self):
        return 100 # Dummy length for testing

    def __getitem__(self, idx):
        # 1. Load Grayscale Images
        # In real code: Load from self.file_list[idx]
        # Simulating data for architecture verification:
        left = torch.randn(1, 480, 640) 
        right = torch.randn(1, 480, 640)
        
        targets = {}
        
        if self.task == 'stereo':
            # GT Disparity
            targets['disp'] = torch.rand(480, 640) * 192.0
            targets['seg'] = None
            targets['det'] = None
            
        elif self.task == 'robotics':
            # TartanAir Seg + Det
            targets['disp'] = None # or sparse
            targets['seg'] = torch.randint(0, 6, (480, 640)).long()
            targets['det'] = torch.zeros((10, 5)) # [x,y,w,h,cls]
            
        return {'left': left, 'right': right, 'targets': targets}

# Test the shape
ds = FusedHexapodDataset("dummy", task='stereo')
sample = ds[0]
print(f"📦 Data Shape Check -> Left: {sample['left'].shape}")

In [ ]:
# === CELL 3: MODEL ARCHITECTURE (FINAL STEREO) ===
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

# --- Helper Blocks ---
class ConvMean(nn.Module):
    """ Replaces mean() with 1x1 Conv for NPU compatibility """
    def __init__(self, in_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, 1, 1, bias=False)
        with torch.no_grad():
            self.conv.weight.fill_(1.0 / in_channels)
        self.conv.weight.requires_grad = False
    def forward(self, x): return self.conv(x)

class ResBlock(nn.Module):
    """ Standard ResBlock for Refinement """
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(channels, channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
    def forward(self, x):
        return self.relu(x + self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))))

# --- Stereo Components ---
class SimpleCorrelation(nn.Module):
    """ Loop-based Correlation + Immediate Reduction (Low Memory) """
    def __init__(self, in_channels, max_disp):
        super().__init__()
        self.D = max_disp
        self.inter_ch = 16
        self.reduce = nn.Sequential(
            nn.Conv2d(in_channels, self.inter_ch, 1, bias=False),
            nn.BatchNorm2d(self.inter_ch), nn.ReLU(inplace=True)
        )
        self.reducer = ConvMean(self.inter_ch) 

    def forward(self, left, right):
        l, r = self.reduce(left), self.reduce(right)
        cost_stack = []
        for d in range(self.D):
            if d > 0:
                # Correlation: Left vs Right(shifted)
                # Pad LEFT to maintain W dimension (Standard stereo convention)
                sim = self.reducer(l[:,:,:,d:] * r[:,:,:,:-d])
                cost_stack.append(F.pad(sim, (d, 0, 0, 0)))
            else:
                cost_stack.append(self.reducer(l * r))
        return torch.cat(cost_stack, dim=1)

class GaussianGating(nn.Module):
    """ Gate = exp(-0.5 * (dist/sigma)^2) """
    def __init__(self, max_disp, gate_range=12):
        super().__init__()
        self.sigma = gate_range / 2.0
        self.disp_coords = nn.Parameter(
            torch.arange(max_disp).float().view(1, max_disp, 1, 1), 
            requires_grad=False
        )

    def forward(self, cost_volume, coarse_disp_s4):
        dist = self.disp_coords - coarse_disp_s4
        return cost_volume * torch.exp(-0.5 * (dist / self.sigma)**2)

# --- Main Stereo Head ---
class StereoHeadGatedV2(nn.Module):
    def __init__(self, ch_s16, ch_s4, max_disp_s4=48):
        super().__init__()
        self.D = max_disp_s4
        
        # 1. Coarse Stage (Stride 16)
        self.D_coarse = self.D // 4 # 12
        self.corr_coarse = SimpleCorrelation(ch_s16, self.D_coarse)
        self.refine_coarse = nn.Sequential(
            nn.Conv2d(self.D_coarse, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            ResBlock(32), ResBlock(32),
            nn.Conv2d(32, self.D_coarse, 3, 1, 1, bias=False)
        )
        self.coarse_sum = nn.Conv2d(self.D_coarse, 1, 1, bias=False) 
        with torch.no_grad():
            self.coarse_sum.weight.data = torch.arange(self.D_coarse).float().view(1, -1, 1, 1)
        
        self.scale_bias = nn.Parameter(torch.zeros(1, 1, 1, 1)) # Learnable Correction

        # 2. Fine Stage (Stride 4)
        self.corr_fine = SimpleCorrelation(ch_s4, self.D)
        self.gating = GaussianGating(max_disp=self.D, gate_range=12) # Fixed +/- 12px
        self.refine_fine = nn.Sequential(
            nn.Conv2d(self.D, 32, 3, 1, 1, bias=False), nn.BatchNorm2d(32), nn.ReLU(),
            ResBlock(32), ResBlock(32),
            nn.Conv2d(32, self.D, 3, 1, 1, bias=False) 
        )

    def forward(self, l16, r16, l4, r4):
        # Coarse
        c_cost = self.corr_coarse(l16, r16)
        c_cost = self.refine_coarse(c_cost)
        
        # Soft-Argmin (Temp=2.0)
        prob_c = F.softmax(c_cost * 2.0, dim=1)
        d_coarse_low = self.coarse_sum(prob_c)
        
        # Upsample & Bias Correct
        d_coarse_s4 = F.interpolate(d_coarse_low, size=l4.shape[-2:], mode='bilinear') * 4.0
        d_coarse_s4 = d_coarse_s4 + self.scale_bias 
        
        # Fine
        raw_cost = self.corr_fine(l4, r4)
        gated_cost = self.gating(raw_cost, d_coarse_s4)
        final_cost = self.refine_fine(gated_cost)
        
        return c_cost, final_cost, d_coarse_s4

# --- The Fused Brain ---
class FusedHexapodModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Shared Backbone
        self.backbone = timm.create_model('mobilenetv3_large_100', pretrained=True, features_only=True, out_indices=(1, 2, 3, 4))
        # Channels: S4=24, S8=40, S16=112, S32=960
        
        # Stereo Head
        self.stereo = StereoHeadGatedV2(ch_s16=112, ch_s4=24)
        
        # [FUTURE] Segmentation Head (LR-ASPP) will go here
        self.seg_head = nn.Identity() 

        # [FUTURE] YOLO Head will go here
        self.yolo_head = nn.Identity()

    def forward(self, left, right=None):
        x = left.repeat(1, 3, 1, 1) # Grayscale -> RGB
        
        # Extract Features [s4, s8, s16, s32]
        fl = self.backbone(x)
        
        # Stereo Branch
        c_log, f_log, d_c = None, None, None
        if right is not None:
            xr = right.repeat(1, 3, 1, 1)
            fr = self.backbone(xr)
            c_log, f_log, d_c = self.stereo(fl[2], fr[2], fl[0], fr[0])
            
        # [FUTURE] Seg Branch: seg_pred = self.seg_head(fl[0], fl[2])
        seg_pred = None
        
        # [FUTURE] Det Branch: det_preds = self.yolo_head(fl[1], fl[2], fl[3])
        det_pred = None
        
        return c_log, f_log, d_c, seg_pred, det_pred

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
model = FusedHexapodModel().to(DEVICE)
print("✅ Fused Model initialized. Stereo Module verified.")

In [ ]:
# === CELL 4: LOSS FUNCTIONS & POST-PROCESSING ===
import torch.optim as optim
from torch.cuda.amp import GradScaler

class StereoPostProcessor:
    """ Converts NPU Logits -> Disparity Map (CPU side) """
    @staticmethod
    def process(logits, max_disp_s4=48):
        prob = F.softmax(logits, dim=1) 
        coords = torch.arange(max_disp_s4, device=logits.device).float().view(1, -1, 1, 1)
        disp_s4 = torch.sum(prob * coords, dim=1, keepdim=True)
        # Scale 4.0 (Stride 4 -> Full Res)
        return F.interpolate(disp_s4, scale_factor=4, mode='bilinear', align_corners=False) * 4.0

class HexapodLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.processor = StereoPostProcessor()
        # [FUTURE] self.seg_loss = nn.CrossEntropyLoss(...)
        # [FUTURE] self.det_loss = ...
        
    def sobel(self, img):
        kx = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], device=img.device).float().view(1,1,3,3)
        ky = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], device=img.device).float().view(1,1,3,3)
        return torch.sqrt(F.conv2d(img, kx, padding=1)**2 + F.conv2d(img, ky, padding=1)**2 + 1e-6)

    def forward(self, preds, targets):
        c_log, f_log, d_coarse_s4, seg_p, det_p = preds
        t_disp = targets.get('disp')
        total_loss = 0; logs = {}
        
        # --- Stereo Loss ---
        if t_disp is not None and f_log is not None:
            mask = (t_disp > 0) & (t_disp < 192)
            if mask.sum() > 0:
                # 1. Fine Loss (L1 + Sobel)
                disp_pred = self.processor.process(f_log)
                loss_main = F.smooth_l1_loss(disp_pred[mask], t_disp[mask]) + \
                            0.5 * F.l1_loss(self.sobel(disp_pred)[mask], self.sobel(t_disp)[mask])
                
                # 2. Coarse Aux Loss (Supervise the Gating Signal)
                # d_coarse_s4 is already scaled to Stride 4 magnitude (0..48)
                # We interpolate to full res spatial size, then multiply by 4 to get full res magnitude (0..192)
                d_c_full = F.interpolate(d_coarse_s4, size=t_disp.shape[-2:], mode='bilinear') * 4.0
                l_aux = F.smooth_l1_loss(d_c_full[mask], t_disp[mask])
                
                loss = loss_main + 0.4 * l_aux
                total_loss += loss
                logs['stereo'] = loss.item()
                logs['coarse_aux'] = l_aux.item()

        return total_loss, logs

criterion = HexapodLoss().to(DEVICE)
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
scaler = GradScaler()
print("✅ Loss Functions ready.")

In [ ]:
# === CELL 5: TRAINING LOOP ===
def train_epoch(model, optimizer, epoch, stage):
    model.train()
    # Freeze Logic
    if stage == 'coarse':
        for p in model.stereo.refine_fine.parameters(): p.requires_grad = False
        model.stereo.scale_bias.requires_grad = False # Freeze bias in coarse stage
    else:
        for p in model.stereo.refine_fine.parameters(): p.requires_grad = True
        model.stereo.scale_bias.requires_grad = True

    # Dummy Data Loop (Replace with DataLoader)
    print(f"--- Epoch {epoch} Stage: {stage} ---")
    for i in range(5):
        l = torch.randn(2, 1, 480, 640).to(DEVICE)
        r = torch.randn(2, 1, 480, 640).to(DEVICE)
        t = {'disp': torch.abs(torch.randn(2, 480, 640).to(DEVICE)) * 192}
        
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            preds = model(l, r)
            loss, _ = criterion(preds, t)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        if i==0: print(f"Loss: {loss.item():.4f}")

# Example Run
print("🚀 1. Train Coarse..."); train_epoch(model, optimizer, 1, 'coarse')
print("🚀 2. Train Fine..."); train_epoch(model, optimizer, 1, 'fine')

In [ ]:
# === CELL 5.1: STEREO TEST TRAINING (SANITY CHECK) ===
import torch.optim as optim
import time

def run_sanity_check():
    print("🧪 Starting Stereo-Only Sanity Check...")
    
    # 1. Setup Model & Optimizer
    model.train()
    # Freezing logic for Coarse Stage (just to test that logic too)
    for p in model.stereo.refine_fine.parameters(): p.requires_grad = False
    model.stereo.scale_bias.requires_grad = False
    
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    scaler = torch.cuda.amp.GradScaler() # For Mixed Precision
    
    # 2. Dummy Data Generator (Simulates a Batch)
    # Batch=2, 1 Channel, 480x640
    dummy_l = torch.randn(2, 1, 480, 640).to(DEVICE)
    dummy_r = torch.randn(2, 1, 480, 640).to(DEVICE)
    # Dummy Disparity (0 to 192)
    dummy_disp = torch.abs(torch.randn(2, 480, 640).to(DEVICE)) * 192.0
    
    targets = {'disp': dummy_disp}
    
    # 3. Training Step Simulation
    start = time.time()
    
    # Forward
    optimizer.zero_grad()
    with torch.cuda.amp.autocast():
        # The model returns 5 values (c, f, d, seg, det). Seg/Det are None.
        preds = model(dummy_l, dummy_r)
        
        # Loss checks only 'disp'
        loss, logs = criterion(preds, targets)
    
    # Backward
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    
    end = time.time()
    
    # 4. Report
    print(f"✅ Forward/Backward Pass Successful!")
    print(f"⏱️  Time per batch: {(end-start)*1000:.2f} ms")
    print(f"📉 Loss Value: {loss.item():.4f}")
    print(f"📝 Logs: {logs}")
    
    # Check outputs
    c_log, f_log, d_c, _, _ = preds
    print(f"📦 Output Shapes:")
    print(f"   - Coarse Logits: {c_log.shape} (Should be [2, 12, 120, 160])")
    print(f"   - Fine Logits:   {f_log.shape} (Should be [2, 48, 120, 160])")
    print(f"   - Coarse Disp:   {d_c.shape}   (Should be [2, 1, 120, 160])")

# Run it!
run_sanity_check()

In [ ]:
# === CELL 6: EXPORT ===
def export():
    model.eval()
    dummy = torch.randn(1, 1, 480, 640).to(DEVICE)
    try:
        torch.onnx.export(model, (dummy, dummy), "hexapod_final.onnx", 
                          input_names=['left', 'right'], 
                          output_names=['coarse_logits', 'fine_logits', 'coarse_disp'], 
                          opset_version=11)
        print("✅ Export Successful! Ready for Hailo.")
    except Exception as e: print(f"❌ Export Failed: {e}")
export()